In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# torch import 전에 설정
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import gc
import json
import time
import random
import inspect
import warnings
import subprocess
from pathlib import Path
from datetime import datetime
from typing import List, Dict

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*bitsandbytes.*")
warnings.filterwarnings("ignore", message=".*_check_is_size.*")


# ============================================================
# 0. Install packages
# ============================================================

def install_packages():
    packages = [
        "transformers>=4.44.0",
        "accelerate",
        "bitsandbytes",
        "peft",
        "datasets",
        "sentencepiece",
        "huggingface_hub",
    ]

    print("패키지 설치/업데이트 중...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]
    )
    print("패키지 준비 완료")


install_packages()


# ============================================================
# 1. Imports
# ============================================================

import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset as TorchDataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)

from peft import (
    PeftModel,
    PeftConfig,
    prepare_model_for_kbit_training,
)

from huggingface_hub import login


# ============================================================
# 2. Clean memory
# ============================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# 3. Google Drive mount
# ============================================================

try:
    from google.colab import drive
    print("Google Drive 마운트 중...")
    drive.mount("/content/drive")
    print("Google Drive 마운트 완료")
except Exception as e:
    print(f"Drive 마운트 스킵: {e}")


# ============================================================
# 4. Optional Hugging Face login
# ============================================================

HF_TOKEN = os.getenv("HF_TOKEN", "").strip()

if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("Hugging Face 로그인 완료")
    except Exception as e:
        print(f"Hugging Face 로그인 실패. 공개 모델이면 계속 진행 가능: {e}")
else:
    print("HF_TOKEN 없음. 공개 모델이면 그대로 진행 가능")


# ============================================================
# 5. Config
# ============================================================

DEFAULT_BASE_MODEL_ID = "MLP-KTLim/llama-3-Korean-Bllossom-8B"

PROJECT_DIR = Path("/content/AI-Project")
DATASET_DIR = PROJECT_DIR / "dataset"
OUTPUT_ROOT = PROJECT_DIR / "outputs"

LATEST_BLOSSOM_LORA_PATH_FILE = OUTPUT_ROOT / "latest_bllossom_lora_path.txt"

# auto면 latest_bllossom_lora_path.txt 사용
# 특정 adapter를 직접 지정하려면 아래처럼 수정:
# CONTINUE_FROM_LORA_PATH = "/content/drive/MyDrive/AIProject/Gwen_Outputs/bllossom_lora_fast_20260617_123456"
CONTINUE_FROM_LORA_PATH = str(
    PROJECT_DIR / "mdoel" / "blossom_LoRA"
)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = OUTPUT_ROOT / f"bllossom_continued_{timestamp}"
MERGED_DATASET_BACKUP = OUTPUT_DIR / "training_dataset_used.json"

SEED = 42

# 이어학습 세팅
MAX_SEQ_LENGTH = 512
NUM_TRAIN_EPOCHS = 1

# A100 기준. OOM 나면 batch 1로 낮추면 됨.
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

# 이어학습은 기존 adapter를 너무 세게 밀지 않기 위해 1e-4 추천
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

LOGGING_STEPS = 5
SAVE_STEPS = 100
EVAL_STEPS = 100

USE_GRADIENT_CHECKPOINTING = True

# tool response 데이터까지 포함
INCLUDE_RESPONSE_DATASETS = True

# 일부 step만 빠르게 확인하고 싶으면 숫자 지정. 예: 300
MAX_STEPS = None

random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 안 잡혔어. Colab 런타임 GPU 설정 확인해줘.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")


# ============================================================
# 6. Resolve existing Bllossom LoRA path
# ============================================================

def resolve_old_lora_path() -> Path:
    if CONTINUE_FROM_LORA_PATH != "auto":
        path = Path(CONTINUE_FROM_LORA_PATH)
    else:
        if not LATEST_BLOSSOM_LORA_PATH_FILE.exists():
            raise FileNotFoundError(
                f"latest_bllossom_lora_path.txt가 없어: {LATEST_BLOSSOM_LORA_PATH_FILE}\n"
                "CONTINUE_FROM_LORA_PATH에 기존 Bllossom LoRA 경로를 직접 넣어줘."
            )

        path_text = LATEST_BLOSSOM_LORA_PATH_FILE.read_text(encoding="utf-8").strip()
        path = Path(path_text)

    if not path.exists():
        raise FileNotFoundError(f"기존 Bllossom LoRA 경로가 없어: {path}")

    adapter_config = path / "adapter_config.json"

    if not adapter_config.exists():
        raise FileNotFoundError(
            f"adapter_config.json이 없어. LoRA adapter 폴더가 맞는지 확인해줘: {path}"
        )

    return path


OLD_LORA_PATH = resolve_old_lora_path()

print("\n이어학습할 기존 Bllossom LoRA:")
print(OLD_LORA_PATH)


# ============================================================
# 7. Read PEFT config / base model
# ============================================================

peft_config = PeftConfig.from_pretrained(str(OLD_LORA_PATH))

BASE_MODEL_ID = peft_config.base_model_name_or_path or DEFAULT_BASE_MODEL_ID

print("\n기존 adapter의 base model:")
print(BASE_MODEL_ID)

print("\n기존 LoRA 설정:")
try:
    adapter_config_json = json.loads(
        (OLD_LORA_PATH / "adapter_config.json").read_text(encoding="utf-8")
    )

    for key in [
        "r",
        "lora_alpha",
        "target_modules",
        "lora_dropout",
        "bias",
        "task_type",
    ]:
        if key in adapter_config_json:
            print(f"- {key}: {adapter_config_json[key]}")
except Exception as e:
    print(f"adapter_config 출력 실패: {e}")

if (OLD_LORA_PATH / "tokenizer_config.json").exists():
    TOKENIZER_SOURCE = str(OLD_LORA_PATH)
else:
    TOKENIZER_SOURCE = BASE_MODEL_ID

print("\nTokenizer source:")
print(TOKENIZER_SOURCE)


# ============================================================
# 8. Dataset loading
# ============================================================

REQUIRED_KEYS = {
    "instruction",
    "output",
    "action_required",
    "action_trigger",
    "action_argument",
}


def load_json_file(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def extract_items_from_json(data):
    if isinstance(data, list):
        return data

    if isinstance(data, dict) and isinstance(data.get("dataset"), list):
        return data["dataset"]

    return []


def is_valid_item(item: dict) -> bool:
    if not isinstance(item, dict):
        return False

    if not REQUIRED_KEYS.issubset(item.keys()):
        return False

    if not isinstance(item["instruction"], str):
        return False

    if not isinstance(item["output"], str):
        return False

    if not isinstance(item["action_required"], bool):
        return False

    if not isinstance(item["action_trigger"], str):
        return False

    if not isinstance(item["action_argument"], (str, int)):
        return False

    if len(item["instruction"].strip()) < 1:
        return False

    if len(item["output"].strip()) < 1:
        return False

    return True


def clean_item(item: dict, source_file: str) -> dict:
    action_trigger = str(item["action_trigger"]).strip() or "NONE"
    action_argument = str(item["action_argument"]).strip()
    action_required = bool(item["action_required"])

    if action_trigger == "NONE":
        action_required = False
        action_argument = ""

    if action_trigger != "NONE":
        action_required = True

    # 날씨인데 지역이 없으면 호출 금지
    if action_trigger == "GET_WEATHER" and action_argument == "":
        action_required = False
        action_trigger = "NONE"
        action_argument = ""

    return {
        "instruction": str(item["instruction"]).strip(),
        "output": str(item["output"]).strip(),
        "action_required": action_required,
        "action_trigger": action_trigger,
        "action_argument": action_argument,
        "_source_file": source_file,
    }


def should_use_file(path: Path) -> bool:
    lower_name = path.name.lower()
    lower_path = str(path).lower()

    excluded_keywords = [
        "rejected",
        "log",
        "adapter_config",
        "trainer_state",
        "training_args",
        "run_config",
        "checkpoint",
        "training_dataset_used",
    ]

    if any(keyword in lower_name for keyword in excluded_keywords):
        return False

    if "gwen_outputs" in lower_path:
        return False

    if not INCLUDE_RESPONSE_DATASETS and "response" in lower_name:
        return False

    return True


def find_dataset_files() -> List[Path]:
    if not DATASET_DIR.exists():
        raise FileNotFoundError(f"dataset 폴더가 없어: {DATASET_DIR}")

    files = []

    for path in DATASET_DIR.rglob("*.json"):
        if should_use_file(path):
            files.append(path)

    unique_files = []
    seen = set()

    for path in files:
        resolved = str(path.resolve())
        if resolved not in seen:
            seen.add(resolved)
            unique_files.append(path)

    return sorted(unique_files)


def load_all_datasets() -> List[dict]:
    files = find_dataset_files()

    print("\n탐색된 dataset JSON 파일:")
    for path in files:
        print(f"- {path}")

    if not files:
        raise ValueError(
            f"JSON 데이터 파일을 찾지 못했어. {DATASET_DIR} 안에 *.json 파일을 넣어줘."
        )

    all_items = []
    file_stats = {}

    for path in files:
        try:
            data = load_json_file(path)
            items = extract_items_from_json(data)

            valid_count = 0

            for item in items:
                if is_valid_item(item):
                    cleaned = clean_item(item, source_file=path.name)
                    all_items.append(cleaned)
                    valid_count += 1

            file_stats[str(path)] = valid_count

        except Exception as e:
            print(f"파일 로드 실패: {path} / {e}")

    print("\n파일별 유효 샘플 수:")
    for path, count in file_stats.items():
        print(f"- {Path(path).name}: {count}")

    deduped = []
    seen = set()

    for item in all_items:
        key = json.dumps(
            {
                "instruction": item["instruction"],
                "output": item["output"],
                "action_required": item["action_required"],
                "action_trigger": item["action_trigger"],
                "action_argument": item["action_argument"],
            },
            ensure_ascii=False,
            sort_keys=True,
        )

        if key in seen:
            continue

        seen.add(key)
        deduped.append(item)

    print(f"\n총 샘플 수: {len(all_items)}")
    print(f"중복 제거 후 샘플 수: {len(deduped)}")

    if len(deduped) == 0:
        raise ValueError("학습할 유효 데이터가 0개야. JSON schema를 확인해줘.")

    category_counts = {}

    for item in deduped:
        source_file = item.get("_source_file", "unknown")
        category_counts[source_file] = category_counts.get(source_file, 0) + 1

    print("\n최종 학습 데이터 구성:")
    for file_name, count in sorted(category_counts.items()):
        print(f"- {file_name}: {count}")

    return deduped


raw_items = load_all_datasets()

with open(MERGED_DATASET_BACKUP, "w", encoding="utf-8") as f:
    json.dump(raw_items, f, ensure_ascii=False, indent=2)

print("\n이번 재학습에 사용한 데이터 백업 저장:")
print(MERGED_DATASET_BACKUP)


# ============================================================
# 9. Prompt formatting
# ============================================================

SYSTEM_PROMPT = """
너는 한국어 반말 AI 비서야.
반드시 JSON 하나만 출력해.
JSON 밖에 다른 문장은 쓰지 마.
""".strip()


def build_assistant_json(item: dict) -> str:
    target = {
        "output": item["output"],
        "action_required": item["action_required"],
        "action_trigger": item["action_trigger"],
        "action_argument": item["action_argument"],
    }

    return json.dumps(
        target,
        ensure_ascii=False,
        indent=2,
    )


def build_messages(item: dict) -> List[Dict[str, str]]:
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": item["instruction"],
        },
        {
            "role": "assistant",
            "content": build_assistant_json(item),
        },
    ]


def preview_samples(items: List[dict], n: int = 5):
    print("\n학습 샘플 미리보기:")
    for idx, item in enumerate(items[:n]):
        print("=" * 80)
        print(f"[sample {idx}] source={item.get('_source_file')}")
        print("instruction:")
        print(item["instruction"])
        print("target:")
        print(build_assistant_json(item))


preview_samples(raw_items, n=min(5, len(raw_items)))


# ============================================================
# 10. Tokenizer / Model loading
# ============================================================

print("\n토크나이저 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_SOURCE,
    trust_remote_code=True,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("토크나이저 로드 완료")


print("\nBase model 로드 중...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=bnb_config,
    trust_remote_code=True,
)

base_model.config.use_cache = False

if USE_GRADIENT_CHECKPOINTING:
    base_model.gradient_checkpointing_enable()
    base_model.enable_input_require_grads()

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
)

print("Base model 로드 완료")


# ============================================================
# 11. Load existing LoRA adapter as trainable
# ============================================================

print("\n기존 Bllossom LoRA adapter 로드 중...")

try:
    model = PeftModel.from_pretrained(
        base_model,
        str(OLD_LORA_PATH),
        is_trainable=True,
    )
except TypeError:
    model = PeftModel.from_pretrained(
        base_model,
        str(OLD_LORA_PATH),
    )

    for name, param in model.named_parameters():
        if "lora_" in name:
            param.requires_grad = True

model.config.use_cache = False

print("기존 Bllossom LoRA adapter 로드 완료")

print("\n학습 가능 파라미터:")
model.print_trainable_parameters()


# ============================================================
# 12. Tokenized dataset
# ============================================================

def apply_chat_template(messages: List[Dict[str, str]], add_generation_prompt: bool) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=add_generation_prompt,
            )
        except Exception:
            pass

    # fallback
    parts = []

    for message in messages:
        role = message["role"]
        content = message["content"]

        if role == "system":
            parts.append(f"<|system|>\n{content}")
        elif role == "user":
            parts.append(f"<|user|>\n{content}")
        elif role == "assistant":
            parts.append(f"<|assistant|>\n{content}")

    if add_generation_prompt:
        parts.append("<|assistant|>\n")

    return "\n".join(parts)


class AssistantJsonDataset(TorchDataset):
    def __init__(self, items: List[dict], tokenizer, max_length: int):
        self.items = items
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.filtered_items = self._filter_items(items)

        print(f"\n원본 샘플 수: {len(items)}")
        print(f"토큰 길이 필터 후 샘플 수: {len(self.filtered_items)}")

        if len(self.filtered_items) == 0:
            raise ValueError(
                "토큰 길이 필터 후 샘플이 0개야. MAX_SEQ_LENGTH를 늘리거나 데이터를 줄여야 해."
            )

    def _filter_items(self, items: List[dict]) -> List[dict]:
        kept = []
        dropped = 0
        dropped_by_source = {}

        for item in items:
            full_text = apply_chat_template(
                build_messages(item),
                add_generation_prompt=False,
            )

            tokenized = self.tokenizer(
                full_text,
                add_special_tokens=False,
            )["input_ids"]

            if len(tokenized) <= self.max_length:
                kept.append(item)
            else:
                dropped += 1
                source = item.get("_source_file", "unknown")
                dropped_by_source[source] = dropped_by_source.get(source, 0) + 1

        print(f"길이 초과로 제외된 샘플 수: {dropped}")

        if dropped_by_source:
            print("길이 초과 제외 파일별 통계:")
            for source, count in sorted(dropped_by_source.items()):
                print(f"- {source}: {count}")

        return kept

    def __len__(self):
        return len(self.filtered_items)

    def __getitem__(self, idx):
        item = self.filtered_items[idx]

        user_messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": item["instruction"],
            },
        ]

        full_messages = build_messages(item)

        prompt_text = apply_chat_template(
            user_messages,
            add_generation_prompt=True,
        )

        full_text = apply_chat_template(
            full_messages,
            add_generation_prompt=False,
        )

        prompt_ids = self.tokenizer(
            prompt_text,
            add_special_tokens=False,
        )["input_ids"]

        full_ids = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )["input_ids"]

        labels = full_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        if all(label == -100 for label in labels):
            labels[-1] = full_ids[-1]

        return {
            "input_ids": torch.tensor(full_ids, dtype=torch.long),
            "attention_mask": torch.ones(len(full_ids), dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }


class DataCollatorForCausalLM:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        input_ids = [f["input_ids"] for f in features]
        attention_mask = [f["attention_mask"] for f in features]
        labels = [f["labels"] for f in features]

        input_ids_padded = pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.pad_token_id,
        )

        attention_mask_padded = pad_sequence(
            attention_mask,
            batch_first=True,
            padding_value=0,
        )

        labels_padded = pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )

        return {
            "input_ids": input_ids_padded,
            "attention_mask": attention_mask_padded,
            "labels": labels_padded,
        }


random.shuffle(raw_items)

if len(raw_items) >= 40:
    eval_size = max(4, int(len(raw_items) * 0.05))
elif len(raw_items) >= 20:
    eval_size = 2
else:
    eval_size = 0

eval_items = raw_items[:eval_size]
train_items = raw_items[eval_size:]

if len(train_items) == 0:
    train_items = raw_items
    eval_items = []

print(f"\nTrain samples before token filter: {len(train_items)}")
print(f"Eval samples before token filter: {len(eval_items)}")

train_dataset = AssistantJsonDataset(
    train_items,
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)

eval_dataset = (
    AssistantJsonDataset(
        eval_items,
        tokenizer=tokenizer,
        max_length=MAX_SEQ_LENGTH,
    )
    if len(eval_items) > 0
    else None
)

data_collator = DataCollatorForCausalLM(tokenizer)


# ============================================================
# 13. TrainingArguments helper
# ============================================================

def build_training_args(kwargs: dict) -> TrainingArguments:
    valid_args = inspect.signature(TrainingArguments.__init__).parameters

    removed = []

    for key in list(kwargs.keys()):
        if key not in valid_args:
            removed.append(key)
            kwargs.pop(key)

    if removed:
        print(f"현재 TrainingArguments에서 지원 안 해서 제거한 인자: {removed}")

    return TrainingArguments(**kwargs)


# ============================================================
# 14. Training
# ============================================================

training_args_kwargs = dict(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",
    seed=SEED,
    remove_unused_columns=False,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    dataloader_pin_memory=False,
)

if MAX_STEPS is not None:
    training_args_kwargs["max_steps"] = MAX_STEPS

if eval_dataset is not None:
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
        training_args_kwargs["eval_strategy"] = "steps"
    else:
        training_args_kwargs["evaluation_strategy"] = "steps"

    training_args_kwargs["eval_steps"] = EVAL_STEPS
    training_args_kwargs["do_eval"] = True
else:
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
        training_args_kwargs["eval_strategy"] = "no"
    else:
        training_args_kwargs["evaluation_strategy"] = "no"

    training_args_kwargs["do_eval"] = False

training_args = build_training_args(training_args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

print("\nBllossom 이어학습 시작")
print(f"OLD_LORA_PATH: {OLD_LORA_PATH}")
print(f"BASE_MODEL_ID: {BASE_MODEL_ID}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"NUM_TRAIN_EPOCHS: {NUM_TRAIN_EPOCHS}")
print(f"PER_DEVICE_TRAIN_BATCH_SIZE: {PER_DEVICE_TRAIN_BATCH_SIZE}")
print(f"GRADIENT_ACCUMULATION_STEPS: {GRADIENT_ACCUMULATION_STEPS}")
print(f"LEARNING_RATE: {LEARNING_RATE}")
print(f"INCLUDE_RESPONSE_DATASETS: {INCLUDE_RESPONSE_DATASETS}")
print(f"MAX_STEPS: {MAX_STEPS}")

trainer.train()
print("Bllossom 이어학습 완료")


# ============================================================
# 15. Save updated LoRA adapter
# ============================================================

print("\n업데이트된 Bllossom LoRA adapter 저장 중...")

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

run_config = {
    "mode": "bllossom_continue_training",
    "old_lora_path": str(OLD_LORA_PATH),
    "base_model": BASE_MODEL_ID,
    "output_dir": str(OUTPUT_DIR),
    "dataset_backup": str(MERGED_DATASET_BACKUP),
    "num_samples_total": len(raw_items),
    "num_train_samples_before_filter": len(train_items),
    "num_train_samples_after_filter": len(train_dataset),
    "num_eval_samples_before_filter": len(eval_items),
    "num_eval_samples_after_filter": len(eval_dataset) if eval_dataset is not None else 0,
    "max_seq_length": MAX_SEQ_LENGTH,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_steps": MAX_STEPS,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "include_response_datasets": INCLUDE_RESPONSE_DATASETS,
    "system_prompt": SYSTEM_PROMPT,
}

try:
    adapter_config_json = json.loads(
        (OUTPUT_DIR / "adapter_config.json").read_text(encoding="utf-8")
    )

    run_config["adapter_config"] = adapter_config_json
except Exception:
    pass

with open(OUTPUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

LATEST_BLOSSOM_LORA_PATH_FILE.write_text(str(OUTPUT_DIR), encoding="utf-8")

print("업데이트된 Bllossom LoRA adapter 저장 완료:")
print(OUTPUT_DIR)
print("latest_bllossom_lora_path.txt 갱신:")
print(LATEST_BLOSSOM_LORA_PATH_FILE)


# ============================================================
# 16. Quick inference test
# ============================================================

print("\n간단 테스트 시작")

model.eval()


def generate_test(user_text: str, max_new_tokens: int = 256):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_text,
        },
    ]

    prompt = apply_chat_template(
        messages,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]

    text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return text.strip()


def parse_model_json(text: str):
    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and start < end:
        try:
            return json.loads(text[start:end + 1])
        except Exception:
            return None

    return None


def normalize_decision(data):
    if not isinstance(data, dict):
        return {
            "output": "",
            "action_required": False,
            "action_trigger": "NONE",
            "action_argument": "",
            "parse_failed": True,
        }

    output = str(data.get("output", "")).strip()
    action_required = data.get("action_required", False)
    action_trigger = str(data.get("action_trigger", "NONE")).strip() or "NONE"
    action_argument = str(data.get("action_argument", "")).strip()

    if isinstance(action_required, str):
        action_required = action_required.lower() == "true"

    if action_trigger == "NONE":
        action_required = False
        action_argument = ""

    if action_trigger != "NONE":
        action_required = True

    if action_trigger == "GET_WEATHER" and action_argument == "":
        action_required = False
        action_trigger = "NONE"
        action_argument = ""

    return {
        "output": output,
        "action_required": action_required,
        "action_trigger": action_trigger,
        "action_argument": action_argument,
    }


test_inputs = [
    "안녕!",
    "나 왔다",
    "오늘 서울 날씨 어때?",
    "오늘 날씨 어때?",
    "비 올까?",
    "부산 비 와?",
    "지금 몇 시야?",
    "오늘 날짜 뭐야?",
    "오늘 무슨 요일이야?",
    "10분 타이머 맞춰줘",
    "그냥 심심해서 말 걸었어",
    "사용자 질문: 지금 몇 시야?\n시간 도구 결과: 현재 시간=14:40, 날짜=2026-09-11, 요일=금요일, 시간대=Asia/Seoul",
    "사용자 질문: 오늘 서울 날씨 어때?\n날씨 API 결과: 지역=서울, 시간=현재, 상태=흐림, 기온=18도, 체감온도=17도, 강수확률=60%, 습도=70%, 바람=약간",
    "사용자 질문: 오늘 서울 날씨 어때?\n액션 실행 결과: action_trigger=GET_WEATHER, status=ERROR, error_code=TIMEOUT, retryable=true",
]

for text in test_inputs:
    print("=" * 80)
    print("USER:")
    print(text)

    raw = generate_test(text)

    print("RAW MODEL:")
    print(raw)

    parsed = parse_model_json(raw)
    normalized = normalize_decision(parsed)

    print("NORMALIZED:")
    print(json.dumps(normalized, ensure_ascii=False, indent=2))

print("\n전부 완료")
print("실행 코드에서 쓸 새 Bllossom LoRA 경로:")
print(OUTPUT_DIR)